In [1]:
from processors import CutflowProcessor, DDTProcessor, TestProcessor
import os, sys, subprocess
import uproot
import numpy as np
from coffea import processor, util
import hist
import awkward as ak
from coffea.nanoevents import NanoEventsFactory
from processors.schema import ScoutingNanoAODSchema, ScoutingJMENanoAODSchema

In [ ]:
fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/data/filter/ScoutingPFRun3/crab_ScoutingPFRun3_Run2022E_359037-359764/231027_150620/0000/nano-wele-chs_62.root"
# fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/mc/hbb/GluGluHto2B_PT-200_M-125_TuneCP5_13p6TeV_powheg-minlo-pythia8/postEE/231026_171043/0000/nanogen_1.root"
# fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/data/ScoutingPFRun3/crab_ScoutingPFRun3_Run2022E_359037-359764/231116_133654/0000/nano_62.root"

events = NanoEventsFactory.from_root(
    fname,
    schemaclass=ScoutingNanoAODSchema,
).events()

print(f"Number of events: {len(events)}")
print(f"Set of runs: {set(events.run)}")
print(f"Number of AK8 jets: {ak.sum(ak.num(events.ScoutingFatJet))}")
trig_bool = ak.all(events.L1["SingleJet180"] | events.L1["HTT360er"] | events.HLT["Mu50"])
print(f"Trigger filter applied: {trig_bool}")
trig_bool = ak.sum(events.L1["SingleJet180"] | events.L1["HTT360er"]) / len(events) * 100
print(f"Percentage of events that pass trigger selection: {trig_bool:.1f} %")

In [ ]:
# fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/data/filter/ScoutingPFRun3/crab_ScoutingPFRun3_Run2022E_359037-359764/231027_150620/0000/nano-wele-chs_62.root"
# fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/mc/hbb/GluGluHto2B_PT-200_M-125_TuneCP5_13p6TeV_powheg-minlo-pythia8/postEE/231026_171043/0000/nanogen_1.root"
fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/data/ScoutingPFRun3/crab_ScoutingPFRun3_Run2022E_359037-359764/231116_133654/0000/nano_62.root"

events = NanoEventsFactory.from_root(
    fname,
    schemaclass=ScoutingNanoAODSchema,
).events()

print(f"Number of events: {len(events)}")
print(f"Set of runs: {set(events.run)}")
print(f"Number of AK8 jets: {ak.sum(ak.num(events.ScoutingFatJet))}")
trig_bool = ak.all(events.L1["SingleJet180"] | events.L1["HTT360er"] | events.HLT["Mu50"])
print(f"Trigger filter applied: {trig_bool}")
trig_bool = ak.sum(events.L1["SingleJet180"] | events.L1["HTT360er"]) / len(events) * 100
print(f"Percentage of events that pass trigger selection: {trig_bool:.1f} %")

In [4]:
from coffea.lumi_tools import LumiList

In [3]:
# import json

# fileset = {}
# with open(f"inputfiles/Run3Summer22EE/ScoutingPFRun3_Run2022EF_noFilter.json") as fin:
#     fileset = json.load(fin)

# fileset = {
#     "NoFilter" : fileset["ScoutingPFRun3_Run2022E_359037-359764_231116_133654_0000"],
# }

fname = "root://cmsxrootd-kit.gridka.de:1094//store/user/adlintul/scouting/run3/2022/data/ScoutingPFRun3/crab_ScoutingPFRun3_Run2022E_359037-359764/231116_133654/0000/nano_62.root"

futures_run = processor.Runner(
    executor = processor.FuturesExecutor(compression=None, workers=2),
    schema=ScoutingNanoAODSchema,
#     maxchunks=5,
#     chunksize=1000,
)

fileset = {
    "NoFilter" : [
        fname
    ]
}

out = futures_run(
    fileset,
    "Events",
    processor_instance=CutflowProcessor()
)

print(f"Number of events in total: {out['events']}")

h = out['cutflow'].project("cut")

# h = out['cutflow'].project("dataset", "reg", "cut", "genflav", "disc", "pt")[
#     0:len:sum,
#     0:len:sum,
#     :,
#     0:len:sum,
#     0:len:sum,
#     0:len:sum,
# ]

print(f"Events before trigger selection: {h[0].value:.0f}")
print(f"Events after trigger selection: {h[1].value:.0f}")
print(f"Percentage after trigger selection: {h[1].value / h[0].value * 100:.1f} %")

Processing  25% ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/8 [ 0:00:07 < 0:00:17 | 0.4  chunk/s ]
Merging (local) 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 [ 0:00:04 < 0:00:00 | ?   merges/s ]

Traceback (most recent call last):
  File "/cvmfs/sft-nightlies.cern.ch/lcg/views/dev4/Fri/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/coffea/processor/executor.py", line 786, in _processwith
    merged = _watcher(FH, self, reducer, pool)
  File "/cvmfs/sft-nightlies.cern.ch/lcg/views/dev4/Fri/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/coffea/processor/executor.py", line 404, in _watcher
    accumulate(
  File "/cvmfs/sft-nightlies.cern.ch/lcg/views/dev4/Fri/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/coffea/processor/accumulator.py", line 100, in accumulate
    accum = iadd(accum, next(gen))
  File "/cvmfs/sft-nightlies.cern.ch/lcg/views/dev4/Fri/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/coffea/processor/accumulator.py", line 79, in iadd
    a[key] = iadd(a[key], b[key])
  File "/cvmfs/sft-nightlies.cern.ch/lcg/views/dev4/Fri/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/coffea/processor/accumulator.py", line 79, in iadd
    a[key] = iadd(

ValueError: Cannot add accumulators of incompatible type (<class 'coffea.lumi_tools.lumi_tools.LumiList'> vs. <class 'coffea.lumi_tools.lumi_tools.LumiList'>)

In [ ]:
h = out['cutflow'].project("cut")

print(f"Events before trigger selection: {h[0].value:.0f}")
print(f"Events after trigger selection: {h[1].value:.0f}")
print(f"Percentage after trigger selection: {h[1].value / h[0].value * 100:.1f} %")

In [ ]:
from coffea import processor, util
from collections import defaultdict
from hist import Hist

class Processor(processor.ProcessorABC):
    def __init__(self):
        pass

    def process(self, events):
        
        dataset = events.metadata['dataset']
        output = defaultdict()
        output["nevents"] = {
            dataset : len(events),
        }
        
        h = Hist(
            hist.axis.Regular(50, 0, 2, name="dr", label=r"$\Delta R$(b quarks)"),
            hist.axis.Regular(50, 200, 500, name="pt", label=r"$p_T$"),
            hist.axis.Regular(50, -8, -0.5, name="rho", label=r"$\rho=ln(m^2_{reg}/p_T^2)$"),
        )
        
        def normalise(arr):
            return ak.to_numpy(ak.fill_none(arr, np.nan))

        def hbb_disc(sig, bkg):
            return ak.where((sig + bkg == 0), 0, sig / (sig + bkg))
        
        fatjets = events.ScoutingFatJet
        fatjets = fatjets[
            (fatjets.pt > 200) &
            (fatjets.pt < 1000) &
            (abs(fatjets.eta) < 2.5)
        ]
        fatjets["qcdrho"] = 2 * np.log(fatjets.particleNet_mass / fatjets.pt)
        fatjet = ak.firsts(fatjets)

        # get Higgs
        H = events.GenPart[
            (abs(events.GenPart.pdgId) == 25)
            & (events.GenPart.hasFlags(['fromHardProcess', 'isLastCopy']))
        ]

        # get b quarks
        b_H = ak.flatten(H.children)

        bpos = b_H[
            (b_H.pdgId == 5)
        ]
        bneg = b_H[
            (b_H.pdgId == -5)
        ]

        h.fill(
            dr = normalise(ak.flatten(bpos.delta_r(bneg))),
            pt = normalise(fatjet.pt),
            rho = normalise(fatjet.qcdrho),
        )
        
        output["h"] = h

        return output

    def postprocess(self, accumulator):
        pass

In [ ]:
import json

fileset = {}
with open(f"inputfiles/Run3Summer22EE/GluGluHto2B.json") as fin:
    fileset = json.load(fin)

futures_run = processor.Runner(
    executor = processor.FuturesExecutor(compression=None, workers=2),
    schema=ScoutingNanoAODSchema,
#     maxchunks=1,
#     chunksize=10,
)

output = futures_run(
    fileset,
    "Events",
    processor_instance=Processor()
)

# util.save(output, "outfiles/2022/tagger/tagger_and_massreg_GluGluHto2B.coffea")

In [ ]:
n1 = output["h"].project("pt", "dr")[
    hist.loc(450):len:sum,
    0:hist.loc(0.8):sum,
]

n2 = output["h"].project("pt", "dr")[
    hist.loc(300):hist.loc(450):sum,
    0:hist.loc(0.8):sum,
]



In [ ]:
n2 / (n2 + n1)

In [ ]:
n2

In [ ]:
n2/n1

In [ ]:
import matplotlib.pyplot as plt
import mplhep
plt.style.use(mplhep.style.CMS)

h = output["h"]
h = h / h.sum()

fig, ax = plt.subplots(figsize=(10, 10))

_, c, _ = mplhep.hist2dplot(h.project("rho", "dr"), cbarsize="4%") # ,cmin=0, cmax=0.5)
ylabel = c.set_label(r"Normalised to unit")

ax.axhline(0.8, linestyle="dashed", color="crimson")

label = mplhep.cms.label(ax=ax, data=False, year="2022", com=13.6, label="Private")

fig.savefig("dr_vs_rho.pdf", bbox_inches='tight', pad_inches=0.1)